# ***Building Dimension Tables***

### ***Importing Libraries***

In [41]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

### ***1. Climate Dimension Table***

###  **Loading the Dataset**

In [42]:
def load_data(file_path):
    """ load the dataset from csv file and return dataframe """
    try:
        data = pd.read_csv(file_path)
    except Exception as e:
        print(f"file not found error: {e}")
        return None
    else:
        print("data loaded successfully")
        return data
    finally:
        print("Execution completed.")

In [43]:
file_path = "../Data/Clean Data/egypt_climate_clean.csv"
climate_data = load_data(file_path)
climate_data.head(3)

data loaded successfully
Execution completed.


,Governorate,Year,Month,Temp_Mean,Temp_Max,Temp_Min,Temp_Range,Surface_Temp,Precipitation,Humidity_2m,Specific_Humidity,Solar_Rad_AllSky,Solar_Rad_ClearSky,Thermal_Rad,Wind_Speed_2m,Wind_Direction_2m,Wind_Speed_10m,Surface_Pressure
0,Cairo,1995,January,11.43,21.41,0.92,12.44,11.29,0.04,64.60,5.26,12.93,14.17,26.20,2.00,318.8,3.07,100.49
1,Cairo,1995,February,13.60,26.23,3.46,12.84,14.04,0.29,63.14,5.75,15.22,17.40,27.23,2.53,342.9,3.76,100.16
2,Cairo,1995,March,16.89,31.16,5.74,15.09,17.99,0.09,51.83,5.53,19.88,21.32,27.81,2.44,355.2,3.57,100.01


#### **data copy**

In [44]:
climate_df = climate_data.copy()

#### **Standardize column names to `snake_case`**

In [45]:
climate_df.columns = (
    climate_df.columns.str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.replace("/","_", regex=False)
    .str.lower())

print(climate_df.columns)

Index(['governorate', 'year', 'month', 'temp_mean', 'temp_max', 'temp_min',
       'temp_range', 'surface_temp', 'precipitation', 'humidity_2m',
       'specific_humidity', 'solar_rad_allsky', 'solar_rad_clearsky',
       'thermal_rad', 'wind_speed_2m', 'wind_direction_2m', 'wind_speed_10m',
       'surface_pressure'],
      dtype='object')


#### **Trim white spaces for all text columns**

In [46]:
text_cols = ['governorate', 'year', 'month', 'temp_mean', 'temp_max', 'temp_min',
            'temp_range', 'surface_temp', 'precipitation', 'humidity_2m',
            'specific_humidity', 'solar_rad_allsky', 'solar_rad_clearsky',
            'thermal_rad', 'wind_speed_2m', 'wind_direction_2m', 'wind_speed_10m','surface_pressure']

for c in text_cols:
    climate_df[c] = climate_df[c].astype(str).str.strip()

#### ***1.1. Dim_Governorate***

In [47]:
dim_governorate = climate_df[["governorate"]].drop_duplicates().sort_values("governorate").reset_index(drop = True)
dim_governorate.insert(0, "governorate_key", dim_governorate.index + 1)
print(dim_governorate.shape)
dim_governorate.head()

(27, 2)


,governorate_key,governorate
0,1,Alexandria
1,2,Aswan
2,3,Asyut
3,4,Bani Suef
4,5,Beheira


#### ***1.2. Dim_Date***

In [48]:
dim_date = climate_df[['year', 'month']].drop_duplicates().reset_index(drop=True)
dim_date.insert(0, "date_key", dim_date.index + 1)

print(dim_date.shape)
dim_date.head(12)

(381, 3)


,date_key,year,month
0,1,1995,January
1,2,1995,February
2,3,1995,March
3,4,1995,April
4,5,1995,May
5,6,1995,June
6,7,1995,July
7,8,1995,August
8,9,1995,September
9,10,1995,October


#### ***1.3. Fact Climate Table***

In [49]:
fact_climate = (climate_df.merge(dim_governorate, on = "governorate", how = "left")\
                .merge(dim_date, on=['year', 'month'], how = "left"))

numeric_cols = ['temp_mean', 'temp_max', 'temp_min',
            'temp_range', 'surface_temp', 'precipitation', 'humidity_2m',
            'specific_humidity', 'solar_rad_allsky', 'solar_rad_clearsky',
            'thermal_rad', 'wind_speed_2m', 'wind_direction_2m', 'wind_speed_10m','surface_pressure']
fact_climate[numeric_cols] = fact_climate[numeric_cols].astype(float)

print(fact_climate.shape)
fact_climate.info()

(10287, 20)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10287 entries, 0 to 10286
Data columns (total 20 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   governorate         10287 non-null  object 
 1   year                10287 non-null  object 
 2   month               10287 non-null  object 
 3   temp_mean           10287 non-null  float64
 4   temp_max            10287 non-null  float64
 5   temp_min            10287 non-null  float64
 6   temp_range          10287 non-null  float64
 7   surface_temp        10287 non-null  float64
 8   precipitation       10287 non-null  float64
 9   humidity_2m         10287 non-null  float64
 10  specific_humidity   10287 non-null  float64
 11  solar_rad_allsky    10287 non-null  float64
 12  solar_rad_clearsky  10287 non-null  float64
 13  thermal_rad         10287 non-null  float64
 14  wind_speed_2m       10287 non-null  float64
 15  wind_direction_2m   10287 non-null  float

In [50]:
fact_climate = (
    fact_climate[[
        "governorate_key", "date_key",
        'temp_mean', 'temp_max', 'temp_min','temp_range', 'surface_temp', 'precipitation', 'humidity_2m',
        'specific_humidity', 'solar_rad_allsky', 'solar_rad_clearsky',
        'thermal_rad', 'wind_speed_2m', 'wind_direction_2m', 'wind_speed_10m','surface_pressure']]
    .sort_values(["governorate_key", "date_key"])
    .reset_index(drop=True))

fact_climate.insert(0, "climate_id", fact_climate.index + 1)

print(fact_climate.shape)
fact_climate.head()

(10287, 18)


,climate_id,governorate_key,date_key,temp_mean,temp_max,temp_min,temp_range,surface_temp,precipitation,humidity_2m,specific_humidity,solar_rad_allsky,solar_rad_clearsky,thermal_rad,wind_speed_2m,wind_direction_2m,wind_speed_10m,surface_pressure
0,1,1,1,13.19,21.16,5.73,8.06,13.31,0.17,70.19,6.40,12.15,13.66,28.10,2.78,276.3,3.88,101.99
1,2,1,2,14.31,23.07,8.10,9.05,14.70,0.73,72.94,7.09,14.25,16.70,28.13,3.06,322.6,4.27,101.70
2,3,1,3,16.44,29.05,9.66,11.17,17.39,0.10,65.19,7.13,18.65,21.38,28.16,2.83,315.4,3.90,101.55
3,4,1,4,18.22,33.34,10.68,11.49,19.64,0.11,59.35,7.19,22.09,24.94,29.42,3.62,349.0,4.85,101.30
4,5,1,5,21.67,35.17,13.21,12.62,23.66,0.00,60.97,9.03,26.43,27.23,30.29,3.29,334.4,4.33,101.31


### ***2. Production Dimension Table***

###  **Loading the Dataset**

In [60]:
file_path = "../Data/Clean Data/Agricultural_Data_Cleaned.csv"
production_data = load_data(file_path)
production_data.head(3)

data loaded successfully
Execution completed.


,Crop,GOV,Year,Session,Area,Production,Yield
0,Wheat,Alexandria,2010,Winter,60424.0,148099.0,2.45
1,Wheat,Beheira,2010,Winter,293263.0,767616.0,2.62
2,Wheat,Gharbiya,2010,Winter,147883.0,390633.0,2.64


#### **data copy**

In [61]:
production_df = production_data.copy()

#### **Standardize column names to `snake_case`**

In [62]:
production_df.columns = (
    production_df.columns.str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.replace("/","_", regex=False)
    .str.lower())

print(production_df.columns)

Index(['crop', 'gov', 'year', 'session', 'area', 'production', 'yield'], dtype='object')


#### **Trim white spaces for all text columns**

In [63]:
text_cols = ['crop', 'gov', 'year', 'session', 'area', 'production', 'yield']

for c in text_cols:
    production_df[c] = production_df[c].astype(str).str.strip()

#### ***2.1. Dim_Crop***

In [64]:
dim_crop = production_df[["crop"]].drop_duplicates().sort_values("crop").reset_index(drop = True)
dim_crop.insert(0, "crop_key", dim_crop.index + 1)
print(dim_crop.shape)
dim_crop.head()

(54, 2)


,crop_key,crop
0,1,Alfalfa
1,2,Apple
2,3,Apricot
3,4,Barley
4,5,Beans


#### ***2.2. Fact Climate Table***

In [65]:
fact_crop_productivity = (production_df.merge(dim_governorate, left_on="gov", right_on="governorate", how="left")\
                            .merge(dim_crop, on="crop", how="left"))

numeric_cols = ['area', 'production', 'yield']
fact_crop_productivity[numeric_cols] = fact_crop_productivity[numeric_cols].astype(float)

print("Shape after merge:", fact_crop_productivity.shape)

Shape after merge: (17430, 10)


In [ ]:
fact_crop_productivity = (
    fact_crop_productivity[[
        "governorate_key", "crop_key", "year",  "area", "production", "yield"]]
    .sort_values(["governorate_key", "crop_key", "year"])
    .reset_index(drop=True))

fact_crop_productivity.insert(0, "crop_productivity_id", fact_crop_productivity.index + 1)

print(fact_crop_productivity.shape)
fact_crop_productivity.head()

(17430, 8)


,crop_productivity_id,governorate_key,crop_key,year,session,area,production,yield
0,1,1,1,2015,Perennial,0.0,0.0,0.0
1,2,1,1,2016,Perennial,0.0,0.0,0.0
2,3,1,1,2017,Perennial,0.0,0.0,0.0
3,4,1,1,2018,Perennial,19.0,380.0,20.0
4,5,1,1,2019,Perennial,33.0,660.0,20.0


### ***3. Water Consumption by GOV Dimension Table***

###  **Loading the Dataset**

In [68]:
file_path = "../Data/Clean Data/Water_Consumption_GOV_clean.csv"
WaterG_data = load_data(file_path)
WaterG_data.head(3)

data loaded successfully
Execution completed.


,Gov,Water_Source,Season,Year,Water_Consumption
0,Luxor,Surface_Water,Winter,2011,70703.0
1,Fayoum,Surface_Water,Winter,2011,784952.0
2,Kafr El Sheikh,Surface_Water,Winter,2011,899221.0


#### **data copy**

In [69]:
WaterG_data = WaterG_data.copy()

#### **Standardize column names to `snake_case`**

In [70]:
WaterG_data.columns = (
    WaterG_data.columns.str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.replace("/","_", regex=False)
    .str.lower())

print(WaterG_data.columns)

Index(['gov', 'water_source', 'season', 'year', 'water_consumption'], dtype='object')


#### **Trim white spaces for all text columns**

In [71]:
text_cols = ['gov', 'water_source', 'season', 'year', 'water_consumption']

for c in text_cols:
    WaterG_data[c] = WaterG_data[c].astype(str).str.strip()

#### ***3.1. Dim_Water_Consumption_GOV***

In [73]:
dim_water = WaterG_data[["water_source","season"]].drop_duplicates().sort_values(["water_source","season"]).reset_index(drop = True)
dim_water.insert(0, "water_key", dim_water.index + 1)
print(dim_water.shape)
dim_water.head()

(8, 3)


,water_key,water_source,season
0,1,Groundwater,Nile season
1,2,Groundwater,Perennial
2,3,Groundwater,Summer
3,4,Groundwater,Winter
4,5,Surface_Water,Nile season


In [76]:
fact_water_gov = (WaterG_data.merge(dim_governorate, left_on="gov", right_on="governorate", how="left")\
                    .merge(dim_water, on=["water_source","season"], how="left"))

fact_water_gov[['water_consumption']] = fact_water_gov[['water_consumption']].astype(float)
print(fact_water_gov.shape)

(1359, 8)


In [77]:
fact_water_gov = (fact_water_gov[["governorate_key", "water_key", "year", "water_consumption"]]
                    .sort_values(["governorate_key", "water_key", "year"])
                    .reset_index(drop=True))

fact_water_gov.insert(0, "water_gov_id", fact_water_gov.index + 1)
print(fact_water_gov.shape)
fact_water_gov.head()

(1359, 5)


,water_gov_id,governorate_key,water_key,year,water_consumption
0,1,1.0,5,2011,9782.0
1,2,1.0,5,2012,36446.0
2,3,1.0,5,2013,35553.0
3,4,1.0,5,2014,37691.0
4,5,1.0,5,2015,36031.0


### ***4. Water Consumption by crops Dimension Table***

###  **Loading the Dataset**

In [80]:
file_path = "../Data/Clean Data/Water_Consumption_Crops_clean.csv"
WaterC_data = load_data(file_path)
WaterC_data.head(3)

data loaded successfully
Execution completed.


,Crop,Season,Year,Water_Consumption
0,Sugar Beet,Winter,2011,618118.0
1,Sugar Beet,Winter,2012,574576.0
2,Sugar Beet,Winter,2013,863085.0


#### **data copy**

In [81]:
WaterC_data = WaterC_data.copy()

#### **Standardize column names to `snake_case`**

In [82]:
WaterC_data.columns = (
    WaterC_data.columns.str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.replace("/","_", regex=False)
    .str.lower())

print(WaterC_data.columns)

Index(['crop', 'season', 'year', 'water_consumption'], dtype='object')


#### **Trim white spaces for all text columns**

In [83]:
text_cols = ['crop', 'season', 'year', 'water_consumption']

for c in text_cols:
    WaterC_data[c] = WaterC_data[c].astype(str).str.strip()

In [89]:
dim_season = WaterC_data[["season"]].drop_duplicates().sort_values("season").reset_index(drop = True)
dim_season.insert(0, "season_key", dim_season.index + 1)
display(dim_season)

,season_key,season
0,1,Nile season
1,2,Summer
2,3,Winter


In [91]:
fact_water_crop = (WaterC_data.merge(dim_crop, on="crop", how="left")\
                    .merge(dim_season, on="season", how="left"))

fact_water_crop[['water_consumption']] = fact_water_crop[['water_consumption']].astype(float)

In [93]:
fact_water_crop = (fact_water_crop[["crop_key", "season_key", "year", "water_consumption"]]
                    .sort_values(["crop_key","year"])
                    .reset_index(drop=True))
fact_water_crop.insert(0, "water_crop_id", fact_water_crop.index + 1)
print(fact_water_crop.shape)
fact_water_crop.head()

(469, 5)


,water_crop_id,crop_key,season_key,year,water_consumption
0,1,1,2,2019,23439.0
1,2,1,2,2020,12460.0
2,3,1,2,2021,15845.0
3,4,1,2,2022,33483.0
4,5,1,2,2023,28753.0


## ***Connecting to SQL Server***

In [125]:
from sqlalchemy import create_engine, text

SQL_SERVER = "localhost"
SQL_DATBASE = "Depi_GP"
SQL_DRIVER = "ODBC Driver 17 for SQL Server"
USE_TRUSTED_CONNECTION = True

try:
    conn_str = (
        f"mssql+pyodbc://@{SQL_SERVER}/{SQL_DATBASE}"
        f"?driver={SQL_DRIVER.replace(' ','+')}&trusted_connection=yes")
    engine = create_engine(conn_str, fast_executemany=True)
    print("Conected to SQL Server")
except Exption as e:
    print(e)
with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION"))
    print(result.fetchone()[0])

Conected to SQL Server
Microsoft SQL Server 2022 (RTM-GDR) (KB5128493) - 16.0.1210.3 (X64) 
	Oct  2 2026 17:33:48 
	Copyright (C) 2022 Microsoft Corporation
	Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 19045: ) (Hypervisor)



## ***Create the schema and tables***

In [126]:
ddl_climate_statements = [
    """
    CREATE TABLE dbo.dim_governorate (
        governorate_key INT PRIMARY KEY,
        governorate VARCHAR(50) NOT NULL);
    """,
    """
    CREATE TABLE dbo.dim_date (
        date_key INT PRIMARY KEY,
        year SMALLINT NOT NULL,
        month VARCHAR(20) NOT NULL);
    """,
    """
    CREATE TABLE dbo.fact_climate (
        climate_id INT PRIMARY KEY,
        governorate_key INT NOT NULL,
        date_key INT NOT NULL,
        temp_mean FLOAT,
        temp_max FLOAT,
        temp_min FLOAT,
        temp_range FLOAT,
        surface_temp FLOAT,
        precipitation FLOAT,
        humidity_2m FLOAT,
        specific_humidity FLOAT,
        solar_rad_allsky FLOAT,
        solar_rad_clearsky FLOAT,
        thermal_rad FLOAT,
        wind_speed_2m FLOAT,
        wind_direction_2m FLOAT,
        wind_speed_10m FLOAT,
        surface_pressure FLOAT,
        CONSTRAINT fk_fact_climate_governorate FOREIGN KEY (governorate_key) REFERENCES dbo.dim_governorate(governorate_key),
        CONSTRAINT fk_fact_climate_date FOREIGN KEY (date_key) REFERENCES dbo.dim_date(date_key));
    """]

with engine.begin() as conn:
    for statement in ddl_climate_statements:
        conn.execute(text(statement))

print("Tables created successfully.")

Tables created successfully.


In [127]:
load_order = [
    ("dim_governorate", dim_governorate),
    ("dim_date", dim_date),
    ("fact_climate", fact_climate)]

for table_name, df in load_order:
    df.to_sql(table_name, con = engine, schema = "dbo", if_exists = "append", 
            index = False, chunksize=1000)

print(f"Loaded {len(df)} rows --into--> dbo.{table_name}")

Loaded 10287 rows --into--> dbo.fact_climate


In [128]:
ddl_other_statements = [
    """
    CREATE TABLE dbo.dim_crop (
        crop_key INT PRIMARY KEY,
        crop VARCHAR(100) NOT NULL);
    """,
    """
    CREATE TABLE dbo.dim_season (
        season_key INT PRIMARY KEY,
        season VARCHAR(50) NOT NULL);
    """,
    """
    CREATE TABLE dbo.fact_crop_productivity (
        crop_productivity_id INT PRIMARY KEY,
        governorate_key INT NOT NULL,
        crop_key INT NOT NULL,
        year SMALLINT NOT NULL,
        area FLOAT,
        production FLOAT,
        yield FLOAT,
        CONSTRAINT fk_fact_crop_gov FOREIGN KEY (governorate_key) REFERENCES dbo.dim_governorate(governorate_key),
        CONSTRAINT fk_fact_crop_crop FOREIGN KEY (crop_key) REFERENCES dbo.dim_crop(crop_key));
    """,
    """
    CREATE TABLE dbo.fact_water_crop (
        water_crop_id INT PRIMARY KEY,
        crop_key INT NOT NULL,
        season_key INT NOT NULL,
        year SMALLINT NOT NULL,
        water_consumption FLOAT,
        CONSTRAINT fk_fact_water_crop_crop FOREIGN KEY (crop_key) REFERENCES dbo.dim_crop(crop_key),
        CONSTRAINT fk_fact_water_crop_season FOREIGN KEY (season_key) REFERENCES dbo.dim_season(season_key));
    """]

with engine.begin() as conn:
    for statement in ddl_other_statements:
        conn.execute(text(statement))

print("Productivity and Water tables created successfully.")

Productivity and Water tables created successfully.


In [129]:
ddl_water_statements = [
    """
    CREATE TABLE dbo.dim_water (
        water_key INT PRIMARY KEY,
        water_source VARCHAR(100) NOT NULL,
        season VARCHAR(50) NOT NULL);
    """,

    """
    CREATE TABLE dbo.fact_water_gov (
        water_gov_id INT PRIMARY KEY,
        governorate_key INT NOT NULL,
        water_key INT NOT NULL,
        year SMALLINT NOT NULL,
        water_consumption FLOAT,
        CONSTRAINT fk_fact_water_gov_gov FOREIGN KEY (governorate_key) REFERENCES dbo.dim_governorate(governorate_key),
        CONSTRAINT fk_fact_water_gov_water FOREIGN KEY (water_key) REFERENCES dbo.dim_water(water_key));
    """
]

with engine.begin() as conn:
    for statement in ddl_water_statements:
        conn.execute(text(statement))

print("Water dimension and fact tables created successfully.")

Water dimension and fact tables created successfully.


In [130]:
load_order = [
    ("dim_crop", dim_crop),
    ("dim_season", dim_season),
    ("dim_water", dim_water),
    ("fact_crop_productivity", fact_crop_productivity),
    ("fact_water_gov", fact_water_gov),
    ("fact_water_crop", fact_water_crop)
]

for table_name, df in load_order:
    df.to_sql(table_name, con=engine, schema="dbo", if_exists="append", 
            index=False, chunksize=1000)
    print(f"Loaded {df.shape[0]} rows --into--> dbo.{table_name}")

Loaded 57 rows --into--> dbo.dim_crop
Loaded 3 rows --into--> dbo.dim_season
Loaded 8 rows --into--> dbo.dim_water
Loaded 17430 rows --into--> dbo.fact_crop_productivity
Loaded 1308 rows --into--> dbo.fact_water_gov
Loaded 469 rows --into--> dbo.fact_water_crop
